# 🛡️ NeuralShield v6.2 — L0–L3 Uçtan Uca (E2E) Katman Testi

**Amaç:** Eğitilmiş **v6.2 mmBERT ONNX** modelini **tam pipeline** içinde (L0→L1→L2→L3→OR-Gate) koşturmak. `verify_full_pipeline.py` **v5** modeliyle çalışıyordu; bu defter **v6.2** ile kapatır.

- L0: `normalize_for_analysis` (deobfuskasyon) → `extended_text`
- L1: `scan_text` (regex) — extended + rot13/reversed/leetspeak varyantları
- L2: `MLSecurityModel.predict` (v6.2 ONNX INT8)
- L3: `scan_and_mask` (DLP)
- Karar: **OR-Gate** = `hard_stego ∨ regex≥60 ∨ ml≥0.70 ∨ (regex ∧ ml=Malicious)`
- Ölçüm: adil benchmark (2.849) + `colab_e2e_suite_stratified.json` üzerinde **Shield ON vs ML-only**; **katman atfı** (hangi katman blokladı).

In [ ]:
# @title 1. Kurulum + repo + v6.2 ONNX dizini
import os, sys, json, shutil, gc
import warnings; warnings.filterwarnings('ignore')
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e: print('drive:', e)
!pip -q install -U transformers onnxruntime scikit-learn langdetect 2>/dev/null
if not os.path.exists('ai-soc-agent-lab'):
    !git clone -q https://github.com/Mustafa-Caliskan/ai-soc-agent-lab.git
if os.path.exists('ai-soc-agent-lab'): os.chdir('ai-soc-agent-lab')
!git pull -q
sys.path.insert(0,'backend')
import normalizer as N
from regex_rules import scan_text
from dlp import scan_and_mask
from ml_model import MLSecurityModel
# v6.2 ONNX dizini hazirla (onnx + tokenizer)
SRC_ONNX='/content/drive/MyDrive/NeuralShield_v6_mmbert/onnx_v6.2/model_sdpa_int8.onnx'
if not os.path.exists(SRC_ONNX):
    SRC_ONNX='/content/drive/MyDrive/NeuralShield_v6_mmbert/onnx_v6.2/model_quantized_int8.onnx'
WD='/content/ns_v62_onnx'; os.makedirs(WD, exist_ok=True)
dst=os.path.join(WD,'model_quantized_int8.onnx')
if os.path.exists(SRC_ONNX) and not os.path.exists(dst): shutil.copy(SRC_ONNX,dst)
if not os.path.exists(os.path.join(WD,'tokenizer_config.json')):
    from transformers import AutoTokenizer
    AutoTokenizer.from_pretrained('jhu-clsp/mmBERT-base').save_pretrained(WD)
print('ONNX:', os.path.exists(dst), '| tokenizer:', os.path.exists(os.path.join(WD,'tokenizer_config.json')))
ML=MLSecurityModel(model_path=WD, sliding_window_enabled=False)
print('ML loaded:', ML.is_loaded)

In [ ]:
# @title 2. Pipeline + OR-Gate + katman atfı (adil benchmark)
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, f1_score
FAIR='/content/drive/MyDrive/NeuralShield_v6_MasterData/heldout_benchmark_v6_fair.json'
data=json.load(open(FAIR,encoding='utf-8'))
texts=[x['text'] for x in data]; y=np.array([int(x['label']) for x in data]); langs=np.array([x['language'] for x in data])

def layer_decide(t):
    n0=N.normalize_for_analysis(t)
    ext=n0.get('extended_text') or n0.get('clean_text') or t
    # L1: extended + varyantlar
    hits=[]; score=0
    for cand in [ext, n0.get('rot13_variant') or '', n0.get('reversed_variant') or '', n0.get('leetspeak_variant') or '']:
        if not cand: continue
        h, m, s = scan_text(cand)
        if h: hits += m
        score=max(score,s)
    regex_hit=bool(hits)
    # L2
    ml=ML.predict(ext)
    mlp=ml['ml_probabilities']['Malicious']
    ml_class=ml.get('ml_class')
    # L3
    dlp_found, masked, dlp_types = scan_and_mask(t)
    # OR-Gate + atif
    layers=[]
    if n0.get('hard_stego_signal'): layers.append('L0')
    if score>=60: layers.append('L1')
    if mlp>=0.70: layers.append('L2')
    if regex_hit and ml_class=='Malicious': layers.append('L1+L2')
    if dlp_found: layers.append('L3')
    blocked = bool(layers)
    return blocked, mlp, layers

res=[]
for t in texts:
    b,p,ls=layer_decide(t); res.append((b,p,ls))
blocked=np.array([r[0] for r in res]); mlp=np.array([r[1] for r in res])
def report(name, pred, score):
    a=pred[y==1]; b=pred[y==0]
    print(f'{name}: Recall={100*a.mean():.1f}% FP={100*b.mean():.1f}% Macro-F1={100*f1_score(y,pred,average="macro"):.1f}% AUC={roc_auc_score(y,score):.4f}')
print('N=',len(data))
report('Shield OR-Gate', blocked, np.maximum(mlp, blocked.astype(float)))
report('ML-only @0.7  ', (mlp>=0.70).astype(int), mlp)
# katman atfi
from collections import Counter
c=Counter()
for b,p,ls in res:
    for l in ls: c[l]+=1
print('\nKatman atfi (blok sayilari):', dict(c))
# per-dil (Shield)
print('\nPer-dil Shield: rec/fp')
for lg in ['en','tr','de','fr','es','it','pt','ru','ar','ja','ko','zh']:
    m=langs==lg
    if not m.any(): continue
    a=blocked[m & (y==1)]; b=blocked[m & (y==0)]
    print(f'  {lg}: {100*a.mean():.1f}% / {100*b.mean():.1f}%')
json.dump({'N':len(data),'layer_counts':dict(c)}, open('/content/drive/MyDrive/NeuralShield_v6_MasterData/e2e_layer_test_v6_2.json','w'), ensure_ascii=False, indent=2)
print('\n-> e2e_layer_test_v6_2.json kaydedildi')

In [ ]:
# @title 3. Tüm Makale Dosyalarını Nihai Altın Pakette Topla (Eksiksiz ZIP)
import os, shutil, glob
OUT_DIR = '/content/neuralshield_paper_pack_final'
os.makedirs(OUT_DIR, exist_ok=True)

# 1. Tüm Doğrulama ve Metrik JSON'ları
search_files = [
    '/content/drive/MyDrive/NeuralShield_v6_MasterData/e2e_layer_test_v6_2.json',
    '/content/drive/MyDrive/NeuralShield_v6_MasterData/native_eval_v6.json',
    '/content/drive/MyDrive/NeuralShield_v6_MasterData/native_subset_v6.json',
    '/content/drive/MyDrive/NeuralShield_v6_MasterData/heldout_benchmark_v6_fair.json',
    '/content/drive/MyDrive/NeuralShield_v6_MasterData/qc_audit_report_v6.json',
    '/content/drive/MyDrive/NeuralShield_v6_MasterData/validation_ci_v6.json',
    '/content/drive/MyDrive/validation_ci_v6.json',
    '/content/validation_ci_v6.json',
    '/content/drive/MyDrive/NeuralShield_v6_mmbert/onnx_v6.2/summary2.json',
    '/content/drive/MyDrive/NeuralShield_v6_mmbert/onnx_v6.2/summary.json'
]
for f in search_files:
    for m in glob.glob(f):
        shutil.copy(m, OUT_DIR)
        print(f'✅ Eklendi: {os.path.basename(m)}')

# 2. ONNX Modelleri
onnx_dir = os.path.join(OUT_DIR, 'onnx')
os.makedirs(onnx_dir, exist_ok=True)
for onnx_f in glob.glob('/content/drive/MyDrive/NeuralShield_v6_mmbert/onnx_v6.2/*.onnx'):
    shutil.copy(onnx_f, onnx_dir)
    print(f'✅ ONNX eklendi: {os.path.basename(onnx_f)}')

# 3. Nihai Makale Paketini ZIP Yap
zip_target = '/content/drive/MyDrive/neuralshield_v6_paper_pack_final'
shutil.make_archive(zip_target, 'zip', OUT_DIR)
print(f'\n🎉 1. Nihai Makale Paketi Hazır:\n📁 {zip_target}.zip')

# 4. Gerçek Şampiyon v6.2 PyTorch Checkpoint'ini ZIP Yap
ck_v62 = '/content/drive/MyDrive/NeuralShield_v6_mmbert/checkpoints_v6_2_backup/checkpoint-6471'
if os.path.exists(ck_v62):
    zip_ck = '/content/drive/MyDrive/neuralshield_v6_checkpoint_v6_2_GOLDEN'
    shutil.make_archive(zip_ck, 'zip', ck_v62)
    print(f'🎉 2. GERÇEK v6.2 ALTIN CHECKPOINT Hazır:\n📁 {zip_ck}.zip')
else:
    print('⚠️ checkpoints_v6_2_backup bulunamadı!')
